# Prompt Tuning or Pre-fix tuning - Manual Way of Soft Prompting

1. Referred from soft prompting research papers to show up the execution in manual way

1. prompt_length (virtual token) - Tells you what to do

2. training data - How to do summarize based on learning

Then the below process begins

[Virtual Tokens] + Input Conversation
                ↓
         t5-small (Frozen)
                ↓
         Generated Summary
                ↓
        Compare with True Summary
                ↓
               Loss
                ↓
   Update [V1 ... V20] ONLY

Initializing the soft prompt module

In [ ]:
import torch
import torch.nn as nn

class SoftPrompt(nn.Module):
    def __init__(self, prompt_length, embedding_dim):
        super().__init__()
        self.prompt = nn.Parameter(torch.randn(prompt_length, embedding_dim))

    def forward(self, batch_size):
        # Expand prompt for batch
        return self.prompt.unsqueeze(0).expand(batch_size, -1, -1)

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load model
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Freeze model
for param in model.parameters():
    param.requires_grad = False

# Create soft prompt
prompt_length = 10  # since it's manual code of development, we don't give any "init_text" as "summarize the following text" instead it takes the random 10 tokens as virtual token from the training data
embedding_dim = model.encoder.embed_tokens.embedding_dim # Capturing the embedding length from the model itself
soft_prompt = SoftPrompt(prompt_length, embedding_dim)

# Optimizer (ONLY soft prompt)
optimizer = torch.optim.Adam(soft_prompt.parameters(), lr=1e-3)

# Sample training data - just one sample
input_text = "summarize: Deep learning models are powerful but require lots of data."
target_text = "Deep learning needs lots of data."

# Tokenize
inputs = tokenizer(input_text, return_tensors="pt")
labels = tokenizer(target_text, return_tensors="pt").input_ids

# Training step
for epoch in range(50):
    optimizer.zero_grad()

    input_ids = inputs.input_ids
    batch_size = input_ids.size(0)

    # Get embeddings
    input_embeds = model.encoder.embed_tokens(input_ids)

    # Add soft prompt
    prompt_embeds = soft_prompt(batch_size)
    inputs_embeds = torch.cat([prompt_embeds, input_embeds], dim=1)

    # Adjust attention mask
    attention_mask = torch.ones(inputs_embeds.shape[:2])

    outputs = model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Inference
with torch.no_grad():
    prompt_embeds = soft_prompt(1)
    input_embeds = model.encoder.embed_tokens(inputs.input_ids)
    inputs_embeds = torch.cat([prompt_embeds, input_embeds], dim=1)

    output_ids = model.generate(inputs_embeds=inputs_embeds)
    print("Summary:", tokenizer.decode(output_ids[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 0, Loss: 1.7318577766418457
Epoch 10, Loss: 1.6655387878417969
Epoch 20, Loss: 1.6005595922470093
Epoch 30, Loss: 1.5329060554504395
Epoch 40, Loss: 1.4587520360946655
Summary: Deep learning models are powerful but require lots of data.


# PEFT Prompting - Library Based Model

1. Adviced for quicker development
2. Faster and cleaner
3. Industry Standard

✅ Yes, you are doing training with dataset and Trainer

❗ But this is NOT normal fine-tuning

👉 This is PEFT-based prompt tuning (parameter-efficient fine-tuning)

---------------------
\
Does 3 important things:

❌ Freezes ALL base model weights

✅ Adds virtual tokens (20 embeddings)

✅ Marks ONLY those tokens as trainable


----------------------------------------------------------

\
🔹 Full Fine-Tuning

Update:
- Attention layers
- Feed-forward layers
- Embeddings
- Everything

👉 Millions/Billions of parameters - Close to 1.1 Billion parameter


\
Our PEFT Prompt Tuning

Update:
[V1, V2, ..., V20]

👉 Only a few thousand parameters - 20 tokens × 2048 dim ≈ ~40K params



---

\
[Virtual Tokens] + Input Conversation
                ↓
         t5-small (Frozen)
                ↓
         Generated Summary
                ↓
        Compare with True Summary
                ↓
               Loss
                ↓
   Update [V1 ... V20] ONLY



---

\
Final Note:

1. You have ONE soft prompt (10 tokens) - > "Summarize the following conversation:"

2. You keep training it again and again using all examples of dataset




In [ ]:
!pip install --upgrade pyarrow datasets transformers peft accelerate bitsandbytes trl -q

# Correcting the runtime restart command for Colab
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)


{'status': 'ok', 'restart': True}

In [ ]:
from peft import PromptTuningConfig, get_peft_model
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Dataset to train the model - To educate how to do the
dataset = load_dataset("knkarthick/samsum")
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# ========================
# Preprocess Function
# ========================

def preprocess(examples):
    prompts = [f"Summarize this dialogue:\n\n{d}\n\nSummary:" for d in examples["dialogue"]]
    inputs = tokenizer(prompts, truncation=True, padding="max_length", max_length=512)
    targets = tokenizer(examples["summary"], truncation=True, padding="max_length", max_length=60)
    inputs["labels"] = targets["input_ids"]
    return inputs

train_data = dataset["train"].map(preprocess, batched=True)

prompt_config = PromptTuningConfig(
    task_type="CAUSAL_LM",
    prompt_tuning_init="TEXT",
    num_virtual_tokens=30,
    tokenizer_name_or_path=tokenizer.name_or_path,
    prompt_tuning_init_text="Summarize the following conversation with the short format:"
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./lora-out",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=10,
    logging_steps=100,
    fp16=True,
    save_total_limit=1,
    report_to="none"
)

soft_prompt_model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").to("cuda")
soft_prompt_model = get_peft_model(soft_prompt_model, prompt_config)

trainer = Trainer(model=soft_prompt_model, args=training_args, train_dataset=train_data, data_collator=collator)
trainer.train()

'''
{
    "stop": [
        "<|start_header_id|>",
        "<|end_header_id|>",
        "<|eot_id|>"
    ]
}
'''

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
100,2.302025
200,2.137414
300,2.097312
400,2.128808
500,2.077713
600,2.069986
700,2.049808
800,2.077573
900,2.082460
1000,2.055266


TrainOutput(global_step=3683, training_loss=2.0693477451429074, metrics={'train_runtime': 2687.0683, 'train_samples_per_second': 5.482, 'train_steps_per_second': 1.371, 'total_flos': 4.681544272497869e+16, 'train_loss': 2.0693477451429074, 'epoch': 1.0})

In [ ]:
import torch

# ========================
# STEP 8: Inference
# ========================

soft_prompt_model.eval()

def generate_summary(text, max_new_tokens=100):
    # IMPORTANT: Keep same instruction style used during training
    prompt = f"Summarize the following conversation:\n{text}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to("cuda")

    with torch.no_grad():
        output_ids = soft_prompt_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.7
        )

    result = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Optional: remove prompt from output
    summary = result.replace(prompt, "").strip()

    return summary


# ========================
# вД Test Inference
# ========================

test_text = """
User: Hey, I had an issue with my order.
Agent: Sorry to hear that, what happened?
User: I received the wrong item.
Agent: I will arrange a replacement for you.
"""

output = generate_summary(test_text)

print("\n===== GENERATED SUMMARY =====\n")
print(output)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")



===== GENERATED SUMMARY =====

User: That's great!
Agent: It will take a bit of time, but I will let you know the estimated time.
User: Thanks a lot!

Summary: Agent apologizes for the wrong order and offers a replacement.


User: How long will it take for the replacement to arrive?
Agent: The estimated time is 3-5 business days, but I can't promise anything.
User: Okay, just make sure it's
